# 05.5 Booleans and Truthiness

Python has `True` and `False`, but the more important idea is **truthiness**:
every object can be used in a condition, and each type decides for itself whether
it counts as true.

This is why `if items:` is the idiomatic way to check for an empty list — and why
`if count:` is a bug waiting to happen when `count` might be zero.

## Theory

### `bool` is a subclass of `int`

This surprises people, but it is literally true:

```python
issubclass(bool, int)   # True
True == 1               # True
True + True             # 2
```

`True` is `1` and `False` is `0`, with a different display. This is a historical
artefact — Python had no `bool` type until version 2.3, and comparisons returned
plain integers. Making `bool` a subclass kept old code working.

It is occasionally useful (`sum(flags)` counts the `True` values) and
occasionally a trap (`True` can be a dict key that collides with `1`).

### Truthiness

Any object can appear in a condition. Python calls `bool()` on it, which:

1. Uses `__bool__()` if the type defines it
2. Otherwise uses `__len__()` — zero length means falsy
3. Otherwise the object is truthy

### The complete list of falsy values

There are not many:

- `False`, `None`
- Zero of any numeric type: `0`, `0.0`, `0j`, `Decimal(0)`, `Fraction(0)`
- Empty collections: `""`, `[]`, `()`, `{}`, `set()`, `range(0)`
- Objects whose `__bool__` returns `False` or whose `__len__` returns `0`

**Everything else is truthy** — including `"False"`, `"0"`, `[0]` and `float("nan")`.

### The distinction that matters

`if x:` and `if x is not None:` are **different checks**. When `x` could
legitimately be `0` or `""`, only the second is correct.

In [ ]:
# bool really is a subclass of int.
print("issubclass(bool, int) ->", issubclass(bool, int))
print("isinstance(True, int) ->", isinstance(True, int))

print("")
print("True and False ARE 1 and 0:")
print("   True == 1   ->", True == 1)
print("   False == 0  ->", False == 0)
print("   True + True ->", True + True)
print("   True * 10   ->", True * 10)
print("   sum([True, False, True]) ->", sum([True, False, True]))

# Which makes counting matches neat.
scores = [45, 82, 91, 30, 77]
passing = sum(score >= 50 for score in scores)
print("")
print("Counting with booleans:")
print("   scores:", scores)
print("   how many >= 50?", passing)

In [ ]:
# The trap: True and 1 are the same dict key.
lookup = {1: "one", True: "true", 0: "zero", False: "false"}

print("A dict built with 1, True, 0 and False as keys:")
print("   result:", lookup)
print("   length:", len(lookup), "<- only two keys survived")

print("")
print("WHY: dict keys are compared by hash and equality. hash(True) ==")
print("hash(1) and True == 1, so the later value overwrote the earlier.")
print("   hash(True):", hash(True), " hash(1):", hash(1))

# Same effect in a set.
print("")
print("Sets behave the same way:")
print("   {1, True, 0, False} ->", {1, True, 0, False})

## Truthiness in practice

In [ ]:
# The complete list of falsy values.
falsy_values = [
    False, None, 0, 0.0, 0j, "", [], (), {}, set(), range(0),
]

print("Everything that is FALSY:")
print("")
for value in falsy_values:
    print(f"   {repr(value):<12} bool() -> {bool(value)}")

print("")
print("That is the whole list. Everything else is truthy.")

In [ ]:
# Values people WRONGLY expect to be falsy.
surprising = [
    ("'False'", "False"),
    ("'0'", "0"),
    ("'None'", "None"),
    ("' '", " "),
    ("[0]", [0]),
    ("[[]]", [[]]),
    ("{'a': None}", {"a": None}),
    ("(0,)", (0,)),
    ("float('nan')", float("nan")),
    ("-1", -1),
    ("0.0001", 0.0001),
]

print("Commonly mistaken for falsy - all of these are TRUTHY:")
print("")
for label, value in surprising:
    print(f"   {label:<14} bool() -> {bool(value)}")

print("")
print("The pattern: any NON-EMPTY string or collection is truthy,")
print("whatever it contains. 'False' is a five-character string.")

### The `0` versus `None` distinction

This is the bug this notebook exists to prevent.

In [ ]:
def describe_wrong(quantity):
    """Check with truthiness - treats 0 as 'not provided'."""
    if quantity:
        return f"quantity is {quantity}"
    return "no quantity given"


def describe_right(quantity):
    """Check for None explicitly - 0 is a real value."""
    if quantity is not None:
        return f"quantity is {quantity}"
    return "no quantity given"


print("Value      Truthiness check         `is not None` check")
print("-" * 62)
for value in [5, 1, 0, None, "", "abc"]:
    print(f"{repr(value):<10} {describe_wrong(value):<24} {describe_right(value)}")

print("")
print("Look at the row for 0. The truthiness version claims no quantity")
print("was given, when 0 was explicitly provided. That is a real bug -")
print("zero items in stock is not the same as unknown stock.")

In [ ]:
# The same trap in default arguments.
def fetch_items_wrong(limit=None):
    """Apply a default when limit is falsy - breaks for limit=0."""
    if not limit:
        limit = 10
    return f"fetching {limit} items"


def fetch_items_right(limit=None):
    """Apply a default only when limit is genuinely absent."""
    if limit is None:
        limit = 10
    return f"fetching {limit} items"


print("Calling with different limits:")
print("")
print("Input      Wrong version              Right version")
print("-" * 62)
for limit in [None, 5, 0]:
    print(f"{str(limit):<10} {fetch_items_wrong(limit):<26} {fetch_items_right(limit)}")

print("")
print("limit=0 should mean 'fetch nothing'. The wrong version silently")
print("fetches 10 instead.")

## Custom truthiness

Your own classes decide their own truthiness via `__bool__` or `__len__`.

In [ ]:
class ShoppingCart:
    """Truthy when it contains items - defined via __len__."""

    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

    def __len__(self):
        # Python falls back to __len__ when __bool__ is absent.
        return len(self.items)


class Account:
    """Truthy when active - defined explicitly via __bool__."""

    def __init__(self, active):
        self.active = active

    def __bool__(self):
        # Explicit control, independent of any length.
        return self.active


cart = ShoppingCart()
print("Empty cart is truthy?", bool(cart))

cart.add("apple")
print("After adding an item:", bool(cart))

print("")
print("Account(active=True) ->", bool(Account(True)))
print("Account(active=False)->", bool(Account(False)))

print("")
print("Resolution order:")
print("   1. __bool__ if defined")
print("   2. otherwise __len__ - zero means falsy")
print("   3. otherwise always truthy")


class NoMethods:
    """Defines neither - so it is always truthy."""


print("")
print("A class with neither method:", bool(NoMethods()))

## Boolean operators return operands, not booleans

`and` and `or` do not return `True`/`False`. They return **one of the operands**,
which is what makes the `or`-default idiom work.

In [ ]:
# `or` returns the first truthy operand, or the last one.
print("`or` returns an operand:")
print("   'a' or 'b'   ->", repr("a" or "b"))
print("   '' or 'b'    ->", repr("" or "b"))
print("   0 or []      ->", repr(0 or []), "<- last one, both falsy")

# `and` returns the first falsy operand, or the last one.
print("")
print("`and` returns an operand:")
print("   'a' and 'b'  ->", repr("a" and "b"))
print("   '' and 'b'   ->", repr("" and "b"))
print("   1 and 2      ->", repr(1 and 2))

# Short-circuiting: the second operand may never be evaluated.
def expensive():
    """Announce that it ran."""
    print("      (expensive() was called)")
    return True


print("")
print("Short-circuiting:")
print("   False and expensive():")
result = False and expensive()
print("      result:", result, "- expensive() was skipped")

print("   True or expensive():")
result = True or expensive()
print("      result:", result, "- expensive() was skipped")

# The common default idiom, and its trap.
print("")
print("The `or` default idiom:")
name = "" or "Anonymous"
print("   '' or 'Anonymous' ->", name)

print("")
print("But it has the same zero trap:")
limit = 0 or 10
print("   0 or 10 ->", limit, "<- 0 was a real value, now lost")
print("   use `limit if limit is not None else 10` instead")

## Takeaways

1. `bool` is a **subclass of `int`** — `True == 1` and `False == 0`, so
   `sum(flags)` counts them.
2. Because of that, `1` and `True` are the **same dict key**.
3. **Falsy:** `False`, `None`, any zero, and any empty collection. Everything
   else is truthy — including `"False"`, `"0"` and `[0]`.
4. `if x:` and `if x is not None:` are **different checks**. Use the second
   whenever `0` or `""` is a legitimate value.
5. Custom classes control truthiness with `__bool__`, falling back to `__len__`.
6. `and` and `or` return **operands**, not booleans, and short-circuit.
7. The `x or default` idiom has the same zero trap — prefer an explicit
   `is None` check.

## Try it yourself

1. Predict `bool()` for: `"0"`, `[]`, `[[]]`, `0.0`, `" "`, `float('nan')`.
2. Build `{0: 'a', False: 'b', 1: 'c', True: 'd'}`. How many keys? Why?
3. Write a function broken by `if not value:` when passed `0`. Then fix it.
4. Write a class that is falsy when a list attribute is empty.
5. Explain why `[] or 'default'` gives `'default'` but `[] and 'x'` gives `[]`.